# Experimento 01: Bag of Words e Binarização

**Objetivo.** Reproduzir, com `scikit-learn`, as matrizes que foram calculadas manualmente em [`docs/representacao-textual.md`](../docs/representacao-textual.md) (seções 2 a 4) e verificar se os resultados reais coincidem com os valores documentados.

**Técnicas estudadas.**

- **Bag of Words (BoW):** cada documento vira um vetor com a contagem de cada termo do vocabulário, sem considerar a ordem.
- **Representação binária:** cada célula indica apenas a presença (1) ou a ausência (0) do termo.

**Corpus (didático, definido na documentação).**

| Documento | Texto |
| --- | --- |
| D1 | *O filme foi excelente.* |
| D2 | *O filme foi bom, mas o final foi fraco.* |
| D3 | *A fotografia é linda e o filme é bom.* |

**Convenção de tokenização.** Os parâmetros padrão do `CountVectorizer` são mantidos: conversão para minúsculas, pontuação descartada, tokens com **pelo menos 2 caracteres** e nenhuma remoção de *stopwords*. Por isso, *o*, *a*, *e* e *é* não entram no vocabulário.

**Escopo.** Este notebook não implementa TF-IDF, n-grams nem modelos de Machine Learning.

## 1. Bibliotecas e versões

In [1]:
import sys

import pandas as pd
import sklearn
from sklearn.feature_extraction.text import CountVectorizer

print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("scikit-learn:", sklearn.__version__)

Python      : 3.12.10
pandas      : 2.3.3
scikit-learn: 1.9.0


## 2. Corpus

O corpus é exatamente o definido na documentação e não deve ser alterado.

In [2]:
ids = ["D1", "D2", "D3"]

corpus = [
    "O filme foi excelente.",
    "O filme foi bom, mas o final foi fraco.",
    "A fotografia é linda e o filme é bom.",
]

for doc_id, texto in zip(ids, corpus):
    print(f"{doc_id}: {texto}")

D1: O filme foi excelente.
D2: O filme foi bom, mas o final foi fraco.
D3: A fotografia é linda e o filme é bom.


## 3. Bag of Words

`CountVectorizer()` com parâmetros padrão. O método `fit_transform` aprende o vocabulário e devolve a matriz documento-termo.

In [3]:
count_vect = CountVectorizer()
bow = count_vect.fit_transform(corpus)

print("Tipo da matriz:", type(bow).__name__)
print("Dimensão      :", bow.shape)

Tipo da matriz: csr_matrix
Dimensão      : (3, 9)


### 3.1 Vocabulário e ordem das colunas

O vocabulário é ordenado alfabeticamente; a posição de cada termo nessa lista é o índice da sua coluna.

In [4]:
termos = count_vect.get_feature_names_out()

print("Tamanho do vocabulário:", len(termos))
for indice, termo in enumerate(termos):
    print(f"coluna {indice}: {termo}")

Tamanho do vocabulário: 9
coluna 0: bom
coluna 1: excelente
coluna 2: filme
coluna 3: final
coluna 4: foi
coluna 5: fotografia
coluna 6: fraco
coluna 7: linda
coluna 8: mas


### 3.2 Tokens considerados em cada documento

O analisador do próprio vetorizador mostra quais tokens são gerados (após minúsculas e descarte de pontuação e de tokens com 1 caractere).

In [5]:
analisador = count_vect.build_analyzer()

tokens_por_doc = {doc_id: analisador(texto) for doc_id, texto in zip(ids, corpus)}

for doc_id, tokens in tokens_por_doc.items():
    print(f"{doc_id} ({len(tokens)} tokens): {tokens}")

D1 (3 tokens): ['filme', 'foi', 'excelente']
D2 (7 tokens): ['filme', 'foi', 'bom', 'mas', 'final', 'foi', 'fraco']
D3 (4 tokens): ['fotografia', 'linda', 'filme', 'bom']


### 3.3 Matriz documento-termo

In [6]:
df_bow = pd.DataFrame(bow.toarray(), index=ids, columns=termos)
df_bow

,bom,excelente,filme,final,foi,fotografia,fraco,linda,mas
D1,0,1,1,0,1,0,0,0,0
D2,1,0,1,1,2,0,1,0,1
D3,1,0,1,0,0,1,0,1,0


### 3.4 Frequência absoluta por termo e tokens por documento

A soma de cada **coluna** é a frequência absoluta do termo no corpus. A soma de cada **linha** é o número de tokens considerados no documento.

In [7]:
freq_absoluta = df_bow.sum(axis=0)
tokens_por_documento = df_bow.sum(axis=1)

print("Frequência absoluta por termo:")
print(freq_absoluta.to_string())
print()
print("Tokens por documento:")
print(tokens_por_documento.to_string())
print()
print("Total de ocorrências:", int(freq_absoluta.sum()))

Frequência absoluta por termo:
bom           2
excelente     1
filme         3
final         1
foi           3
fotografia    1
fraco         1
linda         1
mas           1

Tokens por documento:
D1    3
D2    7
D3    4

Total de ocorrências: 14


## 4. Representação binária

`CountVectorizer(binary=True)`: toda contagem maior que zero passa a valer 1.

In [8]:
bin_vect = CountVectorizer(binary=True)
bow_bin = bin_vect.fit_transform(corpus)

df_bin = pd.DataFrame(bow_bin.toarray(), index=ids, columns=bin_vect.get_feature_names_out())
df_bin

,bom,excelente,filme,final,foi,fotografia,fraco,linda,mas
D1,0,1,1,0,1,0,0,0,0
D2,1,0,1,1,1,0,1,0,1
D3,1,0,1,0,0,1,0,1,0


### 4.1 Frequência documental

Na matriz binária, a soma de cada coluna é a **frequência documental** (em quantos documentos o termo aparece), e não a frequência absoluta.

In [9]:
freq_documental = df_bin.sum(axis=0)

comparacao_freq = pd.DataFrame({
    "freq_absoluta (BoW)": freq_absoluta,
    "freq_documental (binária)": freq_documental,
})
comparacao_freq

,freq_absoluta (BoW),freq_documental (binária)
bom,2,2
excelente,1,1
filme,3,3
final,1,1
foi,3,2
fotografia,1,1
fraco,1,1
linda,1,1
mas,1,1


### 4.2 Diferenças entre BoW e binarização

Lista as células em que as duas matrizes têm valores diferentes.

In [10]:
diferenca = df_bow - df_bin
celulas_diferentes = diferenca[diferenca != 0].stack()

print("Células diferentes:", len(celulas_diferentes))
for (doc_id, termo), _ in celulas_diferentes.items():
    print(f"{doc_id} x '{termo}': BoW = {df_bow.loc[doc_id, termo]}, binária = {df_bin.loc[doc_id, termo]}")

Células diferentes: 1
D2 x 'foi': BoW = 2, binária = 1


## 5. Validação contra a documentação

Os valores esperados abaixo foram transcritos de [`docs/representacao-textual.md`](../docs/representacao-textual.md) (seções 2.2, 3.3 e 4.2). Eles **não** são derivados dos resultados obtidos acima; se alguma verificação falhar, a divergência deve ser investigada, e não os valores esperados ajustados.

In [11]:
# Valores esperados, conforme a documentação
termos_esperados = ["bom", "excelente", "filme", "final", "foi",
                    "fotografia", "fraco", "linda", "mas"]

bow_esperado = [
    [0, 1, 1, 0, 1, 0, 0, 0, 0],  # D1
    [1, 0, 1, 1, 2, 0, 1, 0, 1],  # D2
    [1, 0, 1, 0, 0, 1, 0, 1, 0],  # D3
]

bin_esperado = [
    [0, 1, 1, 0, 1, 0, 0, 0, 0],  # D1
    [1, 0, 1, 1, 1, 0, 1, 0, 1],  # D2 (foi = 1)
    [1, 0, 1, 0, 0, 1, 0, 1, 0],  # D3
]

freq_absoluta_esperada = {"bom": 2, "excelente": 1, "filme": 3, "final": 1, "foi": 3,
                          "fotografia": 1, "fraco": 1, "linda": 1, "mas": 1}

freq_documental_esperada = {"bom": 2, "excelente": 1, "filme": 3, "final": 1, "foi": 2,
                            "fotografia": 1, "fraco": 1, "linda": 1, "mas": 1}

tokens_esperados = [3, 7, 4]

In [12]:
# Vocabulário e dimensões
assert list(termos) == termos_esperados, "Vocabulário diferente do esperado"
assert len(termos) == 9, "O vocabulário deveria ter 9 termos"
assert df_bow.shape == (3, 9), "A matriz BoW deveria ser 3 x 9"
assert df_bin.shape == (3, 9), "A matriz binária deveria ser 3 x 9"
assert list(bin_vect.get_feature_names_out()) == termos_esperados, "Ordem das colunas diferente na versão binária"

# Matrizes completas
assert df_bow.values.tolist() == bow_esperado, "Matriz BoW diferente da documentada"
assert df_bin.values.tolist() == bin_esperado, "Matriz binária diferente da documentada"

# Tokens por documento e frequência total
assert tokens_por_documento.tolist() == tokens_esperados, "Tokens por documento diferentes de 3, 7 e 4"
assert [len(t) for t in tokens_por_doc.values()] == tokens_esperados, "Tokens do analisador diferentes de 3, 7 e 4"
assert int(freq_absoluta.sum()) == 14, "O total de ocorrências deveria ser 14"

# Células não nulas
assert int((df_bow != 0).sum().sum()) == 13, "BoW deveria ter 13 células não nulas"
assert int((df_bin != 0).sum().sum()) == 13, "A matriz binária deveria ter 13 células não nulas"

# Diferença entre BoW e binarização
assert len(celulas_diferentes) == 1, "Deveria haver exatamente 1 célula diferente"
assert df_bow.loc["D2", "foi"] == 2, "BoW (D2, foi) deveria ser 2"
assert df_bin.loc["D2", "foi"] == 1, "Binária (D2, foi) deveria ser 1"

# Frequências por termo
assert freq_absoluta.to_dict() == freq_absoluta_esperada, "Frequência absoluta diferente da documentada"
assert freq_documental.to_dict() == freq_documental_esperada, "Frequência documental diferente da documentada"
assert freq_documental.to_dict() == (df_bow > 0).sum(axis=0).to_dict(), "Soma binária deveria igualar o número de documentos com o termo"

print("Todas as verificações passaram.")

Todas as verificações passaram.


## 6. Interpretação

**O que a matriz representa.** Cada linha é um documento e cada coluna é um termo do vocabulário. Na matriz BoW, a célula contém quantas vezes o termo aparece no documento. Todos os documentos passam a ter vetores do mesmo tamanho (9 posições), o que permite compará-los numericamente.

**O que muda na binarização.** Apenas a célula (D2, *foi*) difere: o termo aparece duas vezes em D2 (*"o filme **foi** bom, mas o final **foi** fraco"*), valor 2 no BoW e 1 na binária. Nas demais, nenhum termo se repete dentro do documento. Com isso, a soma de cada coluna deixa de ser a frequência absoluta e passa a ser a frequência documental (para *foi*: 3 contra 2).

**Que informações são perdidas.**

- **Nas duas representações:** a ordem das palavras, o contexto e o significado. *bom* e *excelente* continuam sendo colunas independentes, e os tokens de um caractere (*o*, *a*, *e*, *é*) foram descartados pelo vetorizador.
- **Somente na binarização:** a repetição do termo, ou seja, a ênfase e a intensidade de uso.

**Por que isso é útil para Machine Learning.** Algoritmos de aprendizado esperam entradas numéricas de tamanho fixo. A matriz documento-termo entrega esse formato: cada linha é um exemplo e cada coluna é um atributo interpretável. A versão por contagem preserva a frequência; a binária é adequada quando a presença do termo importa mais que a repetição. Aqui o corpus tem apenas 3 documentos e nenhum modelo foi treinado, portanto o experimento demonstra a **representação**, e não o seu desempenho em alguma tarefa.